# Colab 1: Environment Setup + Data Preparation
## Reproducing: "Train for the Worst, Plan for the Best" (Kim et al., ICML 2025)
**arXiv:2502.06768v3**

This notebook prepares ALL data needed for the reproduction:
- Google Drive directory structure
- Clone 4 required codebases
- SlimPajama tokenization (GPT-2, L=2048)
- L&O-NAE-SAT synthetic data (m=2, 6 configurations)
- Sudoku dataset from Radcliffe (2020) via Shah et al. (2024)
- Zebra puzzle dataset from Shah et al. (2024)
- LLaDA 8B weight download (deferred)

**Runtime**: CPU or T4 is sufficient. Estimated 2-4 hours.

---
## Cell 1: Mount Google Drive & Create Directory Structure

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = '/content/drive/MyDrive/mdm_reproduction'

dirs = [
    f'{BASE_DIR}/data/slimpajama',
    f'{BASE_DIR}/data/lo_nae_sat',
    f'{BASE_DIR}/data/sudoku/train',
    f'{BASE_DIR}/data/sudoku/test_easy',
    f'{BASE_DIR}/data/sudoku/test_hard',
    f'{BASE_DIR}/data/zebra',
    f'{BASE_DIR}/models',
    f'{BASE_DIR}/results',
    f'{BASE_DIR}/figures',
    f'{BASE_DIR}/codebases',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Directory structure created at:", BASE_DIR)
!ls -R {BASE_DIR}

---
## Cell 2: Install Dependencies

In [ ]:
!pip install torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.36.2 datasets==2.16.1 accelerate==0.25.0
!pip install wandb einops scipy matplotlib seaborn tqdm
# flash-attn is optional; skip if build fails on Colab
!pip install flash-attn --no-build-isolation || echo "flash-attn install failed (optional, continuing)"

---
## Cell 3: Clone Required Codebases

The paper explicitly depends on three codebases plus Shah et al. for datasets:
1. **Nie et al. (2024)** — SMDM: scaling law experiments (Fig 2 Left), MDM training on text
2. **Ye et al. (2024)** — Discrete diffusion: Sudoku/Zebra training and inference (Tables 2, 3, 5)
3. **LLaDA (Nie et al., 2025)** — 8B model experiments (Table 4)
4. **Shah et al. (2024)** — Sudoku/Zebra datasets + ARM-with-ordering baseline

In [ ]:
%cd {BASE_DIR}/codebases

# 1. Nie et al. (2024) — "Scaling up masked diffusion models on text"
#    Used for: scaling law experiments, MDM training on text
!git clone https://github.com/ML-GSAI/SMDM.git 2>/dev/null || echo "SMDM already cloned"

# 2. Ye et al. (2024) — "Beyond Autoregression: Discrete Diffusion for Complex Reasoning and Planning"
#    Used for: Sudoku/Zebra training and inference
#    Paper: "we use the codebase of (Ye et al., 2024) with keeping most of the hyperparameters default"
!git clone https://github.com/HKUNLP/discrete-diffusion.git 2>/dev/null || echo "discrete-diffusion already cloned"

# 3. LLaDA (Nie et al., 2025) — "Large Language Diffusion Models"
#    Used for: Table 4 (LLaDA 8B experiments)
!git clone https://github.com/ML-GSAI/LLaDA.git 2>/dev/null || echo "LLaDA already cloned"

# 4. Shah et al. (2024) — "Causal language modeling can elicit search and reasoning"
#    Used for: Sudoku/Zebra datasets AND the ARM-with-ordering baseline
!git clone https://github.com/kulinshah98/logic-puzzles.git 2>/dev/null || echo "logic-puzzles already cloned"

print("\nCloned codebases:")
!ls -d {BASE_DIR}/codebases/*/

---
## Cell 4: Prepare SlimPajama Dataset

**Paper reference (Section 3.2)**: "We use the Slimpajama dataset (Soboleva et al., 2023)"

- Tokenizer: GPT-2 (vocab size 50257)
- Sequence length: L = 2048 (Section 3.2: "the sequence length L is 2048")
- Pack documents into fixed-length chunks
- Need enough tokens for IsoFLOP scaling law experiments

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

SEQ_LEN = 2048

# Stream SlimPajama and tokenize into packed sequences of length 2048.
# For full reproduction, use at minimum ~10B tokens.
# 500K sequences = ~1B tokens (adjust target_sequences for your compute budget).

# If cerebras/SlimPajama-627B requires auth, first run:
#   !pip install huggingface_hub -q
#   !huggingface-cli login
# Then visit https://huggingface.co/datasets/cerebras/SlimPajama-627B and accept terms.
# As a fallback, we try the community 6B-token subset (sufficient for prototyping).
try:
    dataset = load_dataset("cerebras/SlimPajama-627B", split="train", streaming=True)
    print("Loaded cerebras/SlimPajama-627B")
except Exception as e:
    print(f"Primary source failed ({e}). Trying DKYoon/SlimPajama-6B fallback...")
    dataset = load_dataset("DKYoon/SlimPajama-6B", split="train", streaming=True)
    print("Loaded DKYoon/SlimPajama-6B (6B-token subset)")

buffer = []
all_sequences = []
count = 0
target_sequences = 500000  # ~1B tokens; increase for full reproduction

print(f"Tokenizing SlimPajama into sequences of length {SEQ_LEN}...")
print(f"Target: {target_sequences} sequences ({target_sequences * SEQ_LEN / 1e9:.1f}B tokens)")

for example in dataset:
    tokens = tokenizer.encode(example['text'])
    buffer.extend(tokens)
    
    while len(buffer) >= SEQ_LEN:
        all_sequences.append(buffer[:SEQ_LEN])
        buffer = buffer[SEQ_LEN:]
        count += 1
        if count % 10000 == 0:
            print(f"  {count} sequences ({count * SEQ_LEN / 1e9:.2f}B tokens)")
        if count >= target_sequences:
            break
    if count >= target_sequences:
        break

all_sequences = np.array(all_sequences, dtype=np.int32)

# Split: last 10K for validation, rest for training
val_size = 10000
train_seqs = all_sequences[:-val_size]
val_seqs = all_sequences[-val_size:]

np.save(f'{BASE_DIR}/data/slimpajama/train_sequences.npy', train_seqs)
np.save(f'{BASE_DIR}/data/slimpajama/val_sequences.npy', val_seqs)

print(f"\nSaved {len(train_seqs)} train + {len(val_seqs)} val sequences")
print(f"Total tokens: {len(all_sequences) * SEQ_LEN / 1e9:.2f}B")
print(f"Train shape: {train_seqs.shape}, Val shape: {val_seqs.shape}")


---
## Cell 5: Generate L&O-NAE-SAT Data

**Definition (Section 3.3)**:
- π = identity permutation
- N latent tokens (uniform over {1,...,m}), P observation tokens
- Each observation O_j = NAE(x_{i1}, x_{i2}, x_{i3}) = 1 - 1[all equal]
- Triples (i1,i2,i3) are randomly chosen once and fixed

**Critical derivation**: m = 2 (binary alphabet for latents)
- Table 1 states naive guessing accuracy = 75%
- P(NAE=1 | random from {1,...,m}^3) = 1 - 1/m^2
- For m=2: 1 - 1/4 = 0.75 = 75%. **Confirmed.**

**Configurations**:
- Fig 2 bottom-right (Appendix C.2.1): (N=20, P=280), padded to 512
- Table 1 (Appendix D.1.1): (N,P) in {(25,275),(30,270),(40,260),(50,250),(100,200)}
- All have L = N+P = 300

**Padding** (Appendix C.2.1): "pad the last 212 tokens with an additional token value of 2"

In [ ]:
import numpy as np
import json
import os

BASE_DIR = '/content/drive/MyDrive/mdm_reproduction'


def generate_lo_nae_sat(N, P, m, num_samples, pad_to=512, seed=42):
    """
    Generate L&O-NAE-SAT dataset.
    
    Vocabulary of generated data:
      Latent positions [0, N): values in {1, ..., m}  (i.e., {1, 2} for m=2)
      Observation positions [N, N+P): values in {0, 1}  (NAE output)
      Padding positions [N+P, pad_to): value = 2
    
    Returns:
        sequences: (num_samples, pad_to) int64 array
        triples: (P, 3) int array — the fixed random triples
        metadata: dict
    """
    rng = np.random.RandomState(seed)
    
    # Fix random triples: each element independently from {0,...,N-1}
    # "randomly chosen (pre-fixed) triples (i1, i2, i3) \in [N]" with replacement
    triples = rng.randint(0, N, size=(P, 3))
    
    L = N + P
    sequences = np.zeros((num_samples, pad_to), dtype=np.int64)
    
    for s in range(num_samples):
        # Latent tokens: uniform from {1, ..., m}
        latents = rng.randint(1, m + 1, size=N)
        sequences[s, :N] = latents
        
        # Observation tokens: deterministic NAE of the triple
        for j in range(P):
            i1, i2, i3 = triples[j]
            all_equal = (latents[i1] == latents[i2]) and (latents[i2] == latents[i3])
            sequences[s, N + j] = 0 if all_equal else 1
        
        # Padding with token value 2
        sequences[s, L:] = 2
    
    metadata = {
        'N': N, 'P': P, 'm': m, 'L': L,
        'pad_to': pad_to, 'seed': seed,
        'num_samples': num_samples,
        'naive_accuracy': 1.0 - 1.0 / (m ** 2)
    }
    
    return sequences, triples, metadata


def generate_test_with_same_triples(N, P, m, triples, num_samples, pad_to=512, seed=123):
    """Generate test data using the SAME fixed triples as training data."""
    rng = np.random.RandomState(seed)
    L = N + P
    sequences = np.zeros((num_samples, pad_to), dtype=np.int64)
    
    for s in range(num_samples):
        latents = rng.randint(1, m + 1, size=N)
        sequences[s, :N] = latents
        for j in range(P):
            i1, i2, i3 = triples[j]
            all_equal = (latents[i1] == latents[i2]) and (latents[i2] == latents[i3])
            sequences[s, N + j] = 0 if all_equal else 1
        sequences[s, L:] = 2
    
    return sequences


# ---- Verify naive accuracy ----
_, _, meta_check = generate_lo_nae_sat(20, 280, m=2, num_samples=100)
print(f"Naive guessing accuracy (analytic): {meta_check['naive_accuracy']:.2%}")
assert abs(meta_check['naive_accuracy'] - 0.75) < 1e-6, "m=2 should give 75% naive accuracy"

# ---- Generate all 6 configurations ----
configs = [
    (20, 280),   # Figure 2 bottom-right (Appendix C.2.1)
    (25, 275),   # Table 1
    (30, 270),   # Table 1
    (40, 260),   # Table 1
    (50, 250),   # Table 1
    (100, 200),  # Table 1
]

for N, P in configs:
    print(f"\nGenerating (N={N}, P={P}), L={N+P} ...")
    
    train_data, triples, meta = generate_lo_nae_sat(
        N, P, m=2, num_samples=100000, pad_to=512, seed=42
    )
    test_data = generate_test_with_same_triples(
        N, P, m=2, triples=triples, num_samples=10000, pad_to=512, seed=123
    )
    
    save_dir = f'{BASE_DIR}/data/lo_nae_sat/N{N}_P{P}'
    os.makedirs(save_dir, exist_ok=True)
    np.save(f'{save_dir}/train.npy', train_data)
    np.save(f'{save_dir}/test.npy', test_data)
    np.save(f'{save_dir}/triples.npy', triples)
    with open(f'{save_dir}/metadata.json', 'w') as f:
        json.dump(meta, f, indent=2)
    
    # Empirical verification of naive accuracy
    rng_verify = np.random.RandomState(999)
    random_preds = rng_verify.randint(0, 2, size=(1000, P))
    actual_obs = test_data[:1000, N:N+P]
    random_acc = (random_preds == actual_obs).mean()
    print(f"  Train: {train_data.shape}, Test: {test_data.shape}")
    print(f"  Empirical random guess accuracy: {random_acc:.4f} (expected ~0.75)")

print("\nAll L&O-NAE-SAT data generated.")

---
## Cell 6: Prepare Sudoku Dataset

**Paper (Appendix D.2)**: "For both Sudoku and Zebra puzzles, we use the dataset provided in Shah et al. (2024)... This dataset is created by filtering the puzzles from (Radcliffe, 2020) that can be solved using a fixed list of 7 strategies."

**Source**: Radcliffe (2020), "3 million Sudoku puzzles with ratings", Kaggle.

**Splits**:
- **Easy (train+test)**: Puzzles solvable by 7 strategies (no backtracking)
- **Hard (Table 5 test)**: Remaining puzzles requiring backtracking (~1M puzzles)

**Representation**: 81 tokens (9×9 row-major), values 1-9 for digits, 0 = mask (empty cell)

In [ ]:
import os
import numpy as np

BASE_DIR = '/content/drive/MyDrive/mdm_reproduction'

# =============================================================================
# STEP 1: Check if Shah et al. repo has pre-processed data
# =============================================================================
shah_repo = f'{BASE_DIR}/codebases/logic-puzzles'
print("Checking Shah et al. (2024) repo for pre-processed data...")
if os.path.exists(shah_repo):
    !find {shah_repo} -name "*.csv" -o -name "*.npy" -o -name "*.pkl" -o -name "*.pt" | head -20
    print(f"\nRepo structure:")
    !ls -la {shah_repo}/
else:
    print("WARNING: Shah et al. repo not found. Clone it in Cell 3.")

# =============================================================================
# STEP 2: Download Radcliffe (2020) Sudoku data from Kaggle
# =============================================================================
raw_dir = f'{BASE_DIR}/data/sudoku/raw'
os.makedirs(raw_dir, exist_ok=True)

raw_csv = f'{raw_dir}/sudoku-3m.csv'
if not os.path.exists(raw_csv):
    print("\nDownloading Sudoku dataset from Kaggle...")
    !pip install kaggle -q
    
    # Set your Kaggle API token as an environment variable.
    # Get yours at: https://www.kaggle.com/settings -> API -> Create New Token
    # Paste ONLY the token value (starts with KGAT_...) below:
    from google.colab import userdata
    try:
        # Try Colab Secrets first (Settings gear -> Secrets -> add KAGGLE_API_TOKEN)
        os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
        print("Using KAGGLE_API_TOKEN from Colab Secrets.")
    except Exception:
        # Fallback: set it manually here (or via kaggle.json)
        if 'KAGGLE_API_TOKEN' not in os.environ:
            # Check for kaggle.json in common locations
            !mkdir -p ~/.kaggle
            for p in ['/content/kaggle.json', '/content/drive/MyDrive/kaggle.json']:
                if os.path.exists(p):
                    !cp {p} ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json
                    print(f"Found kaggle.json at {p}")
                    break
            else:
                print("ERROR: No Kaggle credentials found.")
                print("Set your token in Colab Secrets (recommended):")
                print("  1. Click the key icon in the left sidebar")
                print("  2. Add secret: name=KAGGLE_API_TOKEN, value=your_token")
                print("  3. Toggle 'Notebook access' ON")
                print("  4. Re-run this cell")
                raise RuntimeError("Kaggle credentials not configured")
    
    !kaggle datasets download -d radcliffe/3-million-sudoku-puzzles-with-ratings -p {raw_dir}
    
    # Unzip
    import zipfile, glob
    for zf in glob.glob(f'{raw_dir}/*.zip'):
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(raw_dir)
        print(f"Extracted: {zf}")
    
    # Find and rename the CSV
    csvs = glob.glob(f'{raw_dir}/*.csv')
    if csvs and not os.path.exists(raw_csv):
        os.rename(csvs[0], raw_csv)
    
    if os.path.exists(raw_csv):
        print(f"Sudoku CSV ready at: {raw_csv}")
        !ls -lh {raw_csv}
    else:
        print("ERROR: CSV not found after download. Contents of raw dir:")
        !ls -la {raw_dir}/
else:
    print(f"Sudoku CSV already exists at: {raw_csv}")
    !ls -lh {raw_csv}


In [ ]:
import pandas as pd
import numpy as np
import os

BASE_DIR = '/content/drive/MyDrive/mdm_reproduction'

# =============================================================================
# Sudoku String <-> Token Conversion
# =============================================================================
def sudoku_to_tokens(puzzle_str, solution_str):
    """
    Convert Sudoku strings to token arrays.
    
    puzzle_str: 81 chars, '.' or '0' for empty, '1'-'9' for given clues
    solution_str: 81 chars, all '1'-'9'
    
    Returns:
        clues: np.array of 81 ints (0 for empty, 1-9 for clues)
        solution: np.array of 81 ints (1-9)
    """
    clues = np.array([0 if c in '.0' else int(c) for c in puzzle_str], dtype=np.int64)
    solution = np.array([int(c) for c in solution_str], dtype=np.int64)
    return clues, solution


# =============================================================================
# 7-Strategy Sudoku Solver/Filter
# =============================================================================
# These are the 7 constraint-propagation strategies used by Shah et al. (2024)
# to filter "easy" puzzles. A puzzle is "easy" if it can be fully solved using
# only these strategies (no backtracking / guessing).
#
# Strategies:
#   1. Naked Singles: cell with only one candidate
#   2. Hidden Singles: digit that can only go in one cell within a unit
#   3. Naked Pairs: two cells in a unit sharing exactly two candidates
#   4. Hidden Pairs: two digits that can only appear in two cells in a unit
#   5. Pointing Pairs: candidates in a box restricted to one row/col
#   6. Box/Line Reduction: candidates in a row/col restricted to one box
#   7. Naked Triples: three cells in a unit sharing exactly three candidates

def get_candidates(grid):
    """Return a 9x9 grid of candidate sets for each cell."""
    candidates = [[set() for _ in range(9)] for _ in range(9)]
    for r in range(9):
        for c in range(9):
            if grid[r][c] == 0:
                used = set()
                # Row
                used |= set(grid[r])
                # Column
                used |= {grid[rr][c] for rr in range(9)}
                # Box
                br, bc = 3 * (r // 3), 3 * (c // 3)
                for rr in range(br, br + 3):
                    for cc in range(bc, bc + 3):
                        used.add(grid[rr][cc])
                candidates[r][c] = set(range(1, 10)) - used
    return candidates


def naked_singles(grid, cands):
    """Strategy 1: Cell with exactly one candidate."""
    progress = False
    for r in range(9):
        for c in range(9):
            if grid[r][c] == 0 and len(cands[r][c]) == 1:
                grid[r][c] = cands[r][c].pop()
                progress = True
    return progress


def hidden_singles(grid, cands):
    """Strategy 2: Digit that can only go in one cell within a unit."""
    progress = False
    # Check rows, columns, and boxes
    units = []
    for i in range(9):
        units.append([(i, j) for j in range(9)])          # rows
        units.append([(j, i) for j in range(9)])          # cols
    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            units.append([(br+r, bc+c) for r in range(3) for c in range(3)])  # boxes
    
    for unit in units:
        for d in range(1, 10):
            positions = [(r, c) for r, c in unit if grid[r][c] == 0 and d in cands[r][c]]
            if len(positions) == 1:
                r, c = positions[0]
                grid[r][c] = d
                cands[r][c] = set()
                progress = True
    return progress


def _get_units():
    units = []
    for i in range(9):
        units.append([(i, j) for j in range(9)])
        units.append([(j, i) for j in range(9)])
    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            units.append([(br+r, bc+c) for r in range(3) for c in range(3)])
    return units


def naked_pairs(grid, cands):
    """Strategy 3: Two cells in a unit sharing exactly two candidates."""
    progress = False
    for unit in _get_units():
        empty = [(r, c) for r, c in unit if grid[r][c] == 0]
        for i in range(len(empty)):
            for j in range(i + 1, len(empty)):
                r1, c1 = empty[i]
                r2, c2 = empty[j]
                if len(cands[r1][c1]) == 2 and cands[r1][c1] == cands[r2][c2]:
                    pair = cands[r1][c1]
                    for r, c in empty:
                        if (r, c) != (r1, c1) and (r, c) != (r2, c2):
                            before = len(cands[r][c])
                            cands[r][c] -= pair
                            if len(cands[r][c]) < before:
                                progress = True
    return progress


def hidden_pairs(grid, cands):
    """Strategy 4: Two digits that can only appear in two cells in a unit."""
    progress = False
    for unit in _get_units():
        empty = [(r, c) for r, c in unit if grid[r][c] == 0]
        for d1 in range(1, 10):
            for d2 in range(d1 + 1, 10):
                pos_d1 = {(r, c) for r, c in empty if d1 in cands[r][c]}
                pos_d2 = {(r, c) for r, c in empty if d2 in cands[r][c]}
                if pos_d1 == pos_d2 and len(pos_d1) == 2:
                    pair = {d1, d2}
                    for r, c in pos_d1:
                        before = len(cands[r][c])
                        cands[r][c] &= pair
                        if len(cands[r][c]) < before:
                            progress = True
    return progress


def pointing_pairs(grid, cands):
    """Strategy 5: Candidates in a box restricted to one row/col."""
    progress = False
    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            box_cells = [(br+r, bc+c) for r in range(3) for c in range(3)]
            for d in range(1, 10):
                positions = [(r, c) for r, c in box_cells if grid[r][c] == 0 and d in cands[r][c]]
                if len(positions) < 2:
                    continue
                rows = {r for r, c in positions}
                cols = {c for r, c in positions}
                # All in same row
                if len(rows) == 1:
                    row = rows.pop()
                    for c in range(9):
                        if (row, c) not in set(box_cells) and grid[row][c] == 0:
                            if d in cands[row][c]:
                                cands[row][c].discard(d)
                                progress = True
                # All in same col
                if len(cols) == 1:
                    col = cols.pop()
                    for r in range(9):
                        if (r, col) not in set(box_cells) and grid[r][col] == 0:
                            if d in cands[r][col]:
                                cands[r][col].discard(d)
                                progress = True
    return progress


def box_line_reduction(grid, cands):
    """Strategy 6: Candidates in a row/col restricted to one box."""
    progress = False
    for i in range(9):
        # Row i
        for d in range(1, 10):
            positions = [(i, c) for c in range(9) if grid[i][c] == 0 and d in cands[i][c]]
            if len(positions) < 2:
                continue
            boxes = {(i // 3, c // 3) for _, c in positions}
            if len(boxes) == 1:
                box_r, box_c = boxes.pop()
                for r in range(box_r * 3, box_r * 3 + 3):
                    for c in range(box_c * 3, box_c * 3 + 3):
                        if r != i and grid[r][c] == 0 and d in cands[r][c]:
                            cands[r][c].discard(d)
                            progress = True
        # Col i
        for d in range(1, 10):
            positions = [(r, i) for r in range(9) if grid[r][i] == 0 and d in cands[r][i]]
            if len(positions) < 2:
                continue
            boxes = {(r // 3, i // 3) for r, _ in positions}
            if len(boxes) == 1:
                box_r, box_c = boxes.pop()
                for r in range(box_r * 3, box_r * 3 + 3):
                    for c in range(box_c * 3, box_c * 3 + 3):
                        if c != i and grid[r][c] == 0 and d in cands[r][c]:
                            cands[r][c].discard(d)
                            progress = True
    return progress


def naked_triples(grid, cands):
    """Strategy 7: Three cells in a unit sharing exactly three candidates."""
    progress = False
    from itertools import combinations
    for unit in _get_units():
        empty = [(r, c) for r, c in unit if grid[r][c] == 0 and 1 <= len(cands[r][c]) <= 3]
        for combo in combinations(empty, 3):
            union = set()
            for r, c in combo:
                union |= cands[r][c]
            if len(union) == 3:
                combo_set = set(combo)
                for r, c in unit:
                    if grid[r][c] == 0 and (r, c) not in combo_set:
                        before = len(cands[r][c])
                        cands[r][c] -= union
                        if len(cands[r][c]) < before:
                            progress = True
    return progress


def solve_with_7_strategies(puzzle_str):
    """
    Attempt to solve a Sudoku using only the 7 strategies.
    
    Returns:
        solved: bool — True if fully solved without backtracking
        solve_order: list of (row, col, digit) — order in which cells were filled
    """
    grid = [[0]*9 for _ in range(9)]
    for i, ch in enumerate(puzzle_str):
        if ch not in '.0':
            grid[i // 9][i % 9] = int(ch)
    
    solve_order = []
    strategies = [
        naked_singles, hidden_singles, naked_pairs, hidden_pairs,
        pointing_pairs, box_line_reduction, naked_triples
    ]
    
    max_iters = 200
    for _ in range(max_iters):
        cands = get_candidates(grid)
        
        # Record current empty cells to detect newly filled cells
        empty_before = set()
        for r in range(9):
            for c in range(9):
                if grid[r][c] == 0:
                    empty_before.add((r, c))
        
        if not empty_before:
            return True, solve_order  # Solved!
        
        any_progress = False
        for strategy in strategies:
            if strategy(grid, cands):
                any_progress = True
                # Detect newly filled cells
                for r, c in list(empty_before):
                    if grid[r][c] != 0:
                        solve_order.append((r, c, grid[r][c]))
                        empty_before.discard((r, c))
                cands = get_candidates(grid)  # Refresh after each strategy
        
        if not any_progress:
            break
    
    # Check if solved
    solved = all(grid[r][c] != 0 for r in range(9) for c in range(9))
    return solved, solve_order


print("Sudoku strategy solver defined.")

# Quick test
test_puzzle = "..5...9..1..4...7.8..7.3..2.9.....4..4.1.8..5.....6.1.7..2.9..3.3...5..2..1...7.."
solved, order = solve_with_7_strategies(test_puzzle)
print(f"Test puzzle solvable by 7 strategies: {solved}")
print(f"Solve order length: {len(order)} (empty cells: {test_puzzle.count('.')})")

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import json

BASE_DIR = '/content/drive/MyDrive/mdm_reproduction'

# =============================================================================
# Load and filter the Radcliffe (2020) dataset
# =============================================================================
raw_csv = f'{BASE_DIR}/data/sudoku/raw/sudoku-3m.csv'
if not os.path.exists(raw_csv):
    # Try alternate names
    candidates = [
        f'{BASE_DIR}/data/sudoku/raw/sudoku.csv',
        f'{BASE_DIR}/data/sudoku/raw/puzzles.csv',
    ]
    for c in candidates:
        if os.path.exists(c):
            raw_csv = c
            break
    else:
        print("ERROR: Sudoku CSV not found. Download from https://www.kaggle.com/dsv/1495975")
        print(f"Place it at: {BASE_DIR}/data/sudoku/raw/sudoku-3m.csv")
        raise FileNotFoundError(raw_csv)

print(f"Loading {raw_csv}...")
df = pd.read_csv(raw_csv)
print(f"Total puzzles: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(df.head(2))

# Identify puzzle and solution columns
puzzle_col = 'puzzle' if 'puzzle' in df.columns else df.columns[0]
solution_col = 'solution' if 'solution' in df.columns else df.columns[1]

# Filter using 7-strategy solver
print(f"\nFiltering with 7-strategy solver (this takes ~30-60 min for 3M puzzles)...")

easy_puzzles = []   # Solvable by 7 strategies
hard_puzzles = []   # Not solvable (require backtracking)
solve_orders = []   # For ARM-with-ordering baseline

for idx in tqdm(range(len(df)), desc="Filtering"):
    puzzle_str = str(df.iloc[idx][puzzle_col])
    solution_str = str(df.iloc[idx][solution_col])
    
    if len(puzzle_str) != 81 or len(solution_str) != 81:
        continue
    
    clues, solution = sudoku_to_tokens(puzzle_str, solution_str)
    solved, order = solve_with_7_strategies(puzzle_str)
    
    if solved:
        easy_puzzles.append((clues, solution))
        solve_orders.append(order)
    else:
        hard_puzzles.append((clues, solution))

print(f"\nEasy puzzles (7-strategy solvable): {len(easy_puzzles)}")
print(f"Hard puzzles (require backtracking): {len(hard_puzzles)}")

# =============================================================================
# Create train/test splits
# =============================================================================
rng = np.random.RandomState(42)
easy_indices = rng.permutation(len(easy_puzzles))
test_easy_size = min(10000, len(easy_puzzles) // 10)

train_indices = easy_indices[test_easy_size:]
test_easy_indices = easy_indices[:test_easy_size]

# Save easy train
train_clues = np.array([easy_puzzles[i][0] for i in train_indices])
train_solutions = np.array([easy_puzzles[i][1] for i in train_indices])
train_orders = [solve_orders[i] for i in train_indices]

np.save(f'{BASE_DIR}/data/sudoku/train/clues.npy', train_clues)
np.save(f'{BASE_DIR}/data/sudoku/train/solutions.npy', train_solutions)
import pickle
with open(f'{BASE_DIR}/data/sudoku/train/solve_orders.pkl', 'wb') as f:
    pickle.dump(train_orders, f)

# Save easy test
test_easy_clues = np.array([easy_puzzles[i][0] for i in test_easy_indices])
test_easy_solutions = np.array([easy_puzzles[i][1] for i in test_easy_indices])
np.save(f'{BASE_DIR}/data/sudoku/test_easy/clues.npy', test_easy_clues)
np.save(f'{BASE_DIR}/data/sudoku/test_easy/solutions.npy', test_easy_solutions)

# Save hard test
hard_clues = np.array([p[0] for p in hard_puzzles])
hard_solutions = np.array([p[1] for p in hard_puzzles])
np.save(f'{BASE_DIR}/data/sudoku/test_hard/clues.npy', hard_clues)
np.save(f'{BASE_DIR}/data/sudoku/test_hard/solutions.npy', hard_solutions)

meta = {
    'total_puzzles': len(df),
    'easy_count': len(easy_puzzles),
    'hard_count': len(hard_puzzles),
    'train_count': len(train_indices),
    'test_easy_count': len(test_easy_indices),
    'test_hard_count': len(hard_puzzles),
}
with open(f'{BASE_DIR}/data/sudoku/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f"\nSaved:")
print(f"  Train: {len(train_indices)} puzzles")
print(f"  Test (easy): {len(test_easy_indices)} puzzles")
print(f"  Test (hard): {len(hard_puzzles)} puzzles")
print(f"  Solve orders saved for ARM-with-ordering baseline")

---
## Cell 7: Prepare Zebra (Einstein) Puzzle Dataset

**Paper (Appendix D.2)**: Same source as Sudoku — Shah et al. (2024).

The Zebra puzzle assigns attributes (nationality, color, pet, drink, cigarette) to 5 houses. The exact tokenization follows Shah et al.'s format.

In [ ]:
import os

BASE_DIR = '/content/drive/MyDrive/mdm_reproduction'
shah_repo = f'{BASE_DIR}/codebases/logic-puzzles'

print("Zebra dataset: checking Shah et al. (2024) codebase...")
if os.path.exists(shah_repo):
    print(f"\nRepo contents:")
    !ls -la {shah_repo}/
    print(f"\nLooking for Zebra data files:")
    !find {shah_repo} -iname "*zebra*" -o -iname "*einstein*" 2>/dev/null | head -20
    print(f"\nLooking for data directories:")
    !find {shah_repo} -name "data" -type d 2>/dev/null | head -10
else:
    print("WARNING: Shah et al. repo not found. Ensure it was cloned in Cell 3.")

print("\n" + "="*60)
print("ACTION REQUIRED:")
print("Inspect the Shah et al. repo above to find the Zebra puzzle data.")
print("Copy/convert it to:", f'{BASE_DIR}/data/zebra/')
print("The data format (tokenization) must match their codebase.")
print("="*60)

---
## Cell 8: Download LLaDA 8B Weights (Deferred)

**Paper (Section 4.4)**: "we adapted LLaDA, the 8B MDM model from (Nie et al., 2025)"

This is ~16GB. Only download when you're ready for Colab 6.

In [ ]:
# UNCOMMENT WHEN READY FOR COLAB 6
# WARNING: ~16GB download

# !pip install huggingface_hub
# from huggingface_hub import snapshot_download
# 
# snapshot_download(
#     "GSAI-ML/LLaDA-8B-Base",
#     local_dir=f'{BASE_DIR}/models/llada-8b-base',
#     local_dir_use_symlinks=False
# )
# print("LLaDA 8B weights downloaded.")

print("LLaDA download deferred. Uncomment and run when ready for Colab 6.")

---
## Cell 9: Verification Summary

In [ ]:
import os

BASE_DIR = '/content/drive/MyDrive/mdm_reproduction'

checks = {
    'SlimPajama train': f'{BASE_DIR}/data/slimpajama/train_sequences.npy',
    'SlimPajama val': f'{BASE_DIR}/data/slimpajama/val_sequences.npy',
    'L&O-NAE-SAT (20,280) train': f'{BASE_DIR}/data/lo_nae_sat/N20_P280/train.npy',
    'L&O-NAE-SAT (25,275) train': f'{BASE_DIR}/data/lo_nae_sat/N25_P275/train.npy',
    'L&O-NAE-SAT (30,270) train': f'{BASE_DIR}/data/lo_nae_sat/N30_P270/train.npy',
    'L&O-NAE-SAT (40,260) train': f'{BASE_DIR}/data/lo_nae_sat/N40_P260/train.npy',
    'L&O-NAE-SAT (50,250) train': f'{BASE_DIR}/data/lo_nae_sat/N50_P250/train.npy',
    'L&O-NAE-SAT (100,200) train': f'{BASE_DIR}/data/lo_nae_sat/N100_P200/train.npy',
    'Sudoku train clues': f'{BASE_DIR}/data/sudoku/train/clues.npy',
    'Sudoku train solutions': f'{BASE_DIR}/data/sudoku/train/solutions.npy',
    'Sudoku test (easy)': f'{BASE_DIR}/data/sudoku/test_easy/clues.npy',
    'Sudoku test (hard)': f'{BASE_DIR}/data/sudoku/test_hard/clues.npy',
    'Sudoku solve orders': f'{BASE_DIR}/data/sudoku/train/solve_orders.pkl',
    'SMDM codebase': f'{BASE_DIR}/codebases/SMDM',
    'Discrete-diffusion codebase': f'{BASE_DIR}/codebases/discrete-diffusion',
    'LLaDA codebase': f'{BASE_DIR}/codebases/LLaDA',
    'Shah et al. codebase': f'{BASE_DIR}/codebases/logic-puzzles',
}

print("=" * 60)
print("DATA PREPARATION VERIFICATION")
print("=" * 60)
all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    if not exists:
        all_ok = False
    print(f"  [{status:>7}] {name}")

print("\n" + ("ALL CHECKS PASSED" if all_ok else "SOME ITEMS MISSING — review above"))